In [1]:
import pandas as pd
import requests
from tqdm import tqdm
from rag_pipeline import ScienceRAG
import logging

# Подавляем INFO логи от httpx
logging.getLogger("httpx").setLevel(logging.WARNING)

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0201 11:46:03.223000 26964 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
df = pd.read_json('../../data/gd.json')

In [3]:
rag = ScienceRAG()

INFO:rag_pipeline:Initializing RAG on device: CUDA
INFO:rag_pipeline:Connected to Qdrant collection: nlp2025_chunks
INFO:rag_pipeline:Loading retriever: Qwen/Qwen3-Embedding-0.6B...
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: Qwen/Qwen3-Embedding-0.6B
INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query
INFO:rag_pipeline:Retriever loaded on CPU
INFO:rag_pipeline:Loading LLM: Qwen/Qwen2.5-1.5B-Instruct...
INFO:rag_pipeline:LLM loaded on CUDA
INFO:rag_pipeline:RAG system is ready.



In [4]:
def get_rag_data(question, top_k):
    res = rag.answer(question, top_k)
    context_list = [source['text'] for source in res.get('sources', [])]
    return pd.Series([res['answer'], context_list])

TOP_K = 10

In [5]:
tqdm.pandas()
df[['answer', 'context']] = df['question'].progress_apply(get_rag_data, top_k=TOP_K)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [10:04<00:00, 12.09s/it]


In [18]:
df.to_parquet('../../data/ragas/gd_filled.parquet', index=False)